In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [11]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-2.5-flash",
    model_provider="google_genai",
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [13]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='7697aef7-8ec3-4473-9dd9-902df5e96406'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'8a9a2fc7-2634-40a8-bd8e-0fe640ef444c': 'CqoBAQw51scdjrYhbTofjjNoeRn2crsooh13GbsWnQI3tp/zXkwrkdkBKQsYR5ipYXMUyscpSOdcbRMM+VAonqv8sngT97qXWn70IzvOZHDkuV3I7qllTU2nSO82fofbtvX3dJFBk7fpdihJIQjxmocTMi53hR0g7bUMl2Ixp+zYkRNyRGtG6sIEjua1AnLsvJ85moUV44ahdzaDUOFGeS4d8T/Cnnw7kEnBVkk='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e3ad5-2637-7f61-8904-9249a8fd4046-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '8a9a2fc7-2634-40a8-bd8e-0fe640ef444c', 'type': 'tool_call'}]

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)